In [1]:
# Dépendances du notebook
%pip install openpyxl==3.1.3 pandas==3.0.3 s3fs==2026.3.0 -q


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


# Import des packages nécessaires

In [2]:
import pandas as pd
import io
import os
import openpyxl
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment




# Import des données
Les données sont disponibles sur le site [DATA AMELI](https://data.ameli.fr/pages/home/).




J’ai choisi de travailler avec deux jeux de données totalement distincts :
**Une vue régionale et départementale**, décrivant les patients selon la pathologie, le traitement chronique ou l’épisode de soins, le sexe, la classe d’âge, la région et le département.  
  - Source : [Effectifs – Patients par pathologie](https://data.ameli.fr/explore/dataset/effectifs/information/)

**Une vue nationale**, portant sur les dépenses remboursées par pathologie et par patient en moyenne.  
  - Source : [Dépenses remboursées par pathologie](https://data.ameli.fr/explore/dataset/depenses/information/)

Comme le fichier **effectifs** est particulièrement volumineux, j’ai opté pour une **lecture en chunks**, ce qui permet de filtrer les données au fur et à mesure du chargement et d’éviter une surcharge mémoire.

J’ai ensuite enrichi ces données en les fusionnant avec un second fichier **region_dept** contenant les libellés des régions et des départements, afin d’obtenir un rendu plus harmonieux et plus lisible dans les visualisations.

Pour garantir un chargement fiable et éviter les problèmes liés aux chemins locaux, j’ai préféré déposer mes fichiers CSV sur Onyxia et les récupérer directement via leur URL MinIO. Cette approche assure un accès stable et reproductible aux données, quel que soit l’environnement d’exécution.

### Effectif de patients par pathologie, sexe, classe d'âge et territoire (département, région)

In [22]:
chunks = []

for chunk in pd.read_csv('https://minio.lab.sspcloud.fr/aissataa/PROJET_MEYTE/effectifs.csv', sep=';', chunksize=100_000,low_memory=False):
    filtered = chunk[(chunk['annee'] >= 2021) & (chunk['dept'] != '999')]
    chunks.append(filtered)

df_effectifs = pd.concat(chunks, ignore_index=True)

#Le datasaet qui va servir de jointure pour récuperer les libelles des départements et région
df_regions_dept = pd.read_csv(
    "https://minio.lab.sspcloud.fr/aissataa/PROJET_MEYTE/LibelleRegionDept.csv",
    sep=";",
    encoding="latin1",
    usecols=["departement", "libelle_departement", "libelle_region"]
)

# Harmonisation des colonnes , les 1 devient des 01 
df_regions_dept["departement"] = df_regions_dept["departement"].astype(str).str.zfill(2)
df_effectifs["dept"] = df_effectifs["dept"].astype(str).str.zfill(2)

# Jointure des 2 :
df_fusion = pd.merge(
    df_effectifs, 
    df_regions_dept, 
    left_on="dept", 
    right_on="departement", 
    how="inner"
)



Ici, je n’utilise pas `.copy()` car mon objectif n’est pas de dupliquer le DataFrame, mais simplement de le renommer.  
En faisant :

In [23]:
df_effectifs = df_fusion
del df_fusion # Et par la suite je supprime le dataFrame pour libérer de l'espace

## Nettoyage du dataset : df_effectifs

### Vérification et traitement des doublons

J’affiche ici les lignes du DataFrame qui apparaissent en doublon afin d’identifier
les répétitions éventuelles dans les données. L’option `keep=False` permet de visualiser
toutes les occurrences d’un doublon, ce qui facilite le diagnostic avant nettoyage.

Le fichier des correspondances *région–département* contient plusieurs doublons,ce qui génère des lignes dupliquées après la jointure.
Pour éviter ces répétitions dans le DataFrame final, j’ai décidé de supprimer les doublons avant de poursuivre l’analyse.


In [24]:
df_effectifs[df_effectifs.duplicated(keep=False)]


,annee,patho_niv1,patho_niv2,patho_niv3,top,cla_age_5,sexe,region,dept,Ntop,Npop,prev,Niveau prioritaire,libelle_classe_age,libelle_sexe,tri,libelle_region,departement,libelle_departement
0,2023,Maladies inflammatoires ou rares ou infection VIH,Maladies rares,Mucoviscidose,RAR_MUC_IND,95et+,9,32,59,NaN,8460,NaN,3,plus de 95 ans,tous sexes,78.0,Hauts-de-France,59,Nord
1,2023,Maladies inflammatoires ou rares ou infection VIH,Maladies rares,Mucoviscidose,RAR_MUC_IND,95et+,9,32,59,NaN,8460,NaN,3,plus de 95 ans,tous sexes,78.0,Hauts-de-France,59,Nord
2,2023,Maladies inflammatoires ou rares ou infection VIH,Maladies rares,Mucoviscidose,RAR_MUC_IND,95et+,9,32,59,NaN,8460,NaN,3,plus de 95 ans,tous sexes,78.0,Hauts-de-France,59,Nord
3,2023,Maladies inflammatoires ou rares ou infection VIH,Maladies rares,Mucoviscidose,RAR_MUC_IND,95et+,9,32,59,NaN,8460,NaN,3,plus de 95 ans,tous sexes,78.0,Hauts-de-France,59,Nord
4,2023,Maladies inflammatoires ou rares ou infection VIH,Maladies rares,Mucoviscidose,RAR_MUC_IND,95et+,9,32,59,NaN,8460,NaN,3,plus de 95 ans,tous sexes,78.0,Hauts-de-France,59,Nord
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7317445,2023,Maladies cardioneurovasculaires,Artériopathie périphérique,Artériopathie périphérique,MCV_APE_IND,45-49,2,75,64,40.0,21590,0.185,"2,3",de 45 à 49 ans,femmes,27.0,Nouvelle-Aquitaine,64,Pyrénées-Atlantiques
7317446,2023,Maladies cardioneurovasculaires,Artériopathie périphérique,Artériopathie périphérique,MCV_APE_IND,45-49,2,75,64,40.0,21590,0.185,"2,3",de 45 à 49 ans,femmes,27.0,Nouvelle-Aquitaine,64,Pyrénées-Atlantiques
7317447,2023,Maladies cardioneurovasculaires,Artériopathie périphérique,Artériopathie périphérique,MCV_APE_IND,45-49,2,75,64,40.0,21590,0.185,"2,3",de 45 à 49 ans,femmes,27.0,Nouvelle-Aquitaine,64,Pyrénées-Atlantiques
7317448,2023,Maladies cardioneurovasculaires,Artériopathie périphérique,Artériopathie périphérique,MCV_APE_IND,45-49,2,75,64,40.0,21590,0.185,"2,3",de 45 à 49 ans,femmes,27.0,Nouvelle-Aquitaine,64,Pyrénées-Atlantiques


#### Nombre de doublons par colonne

In [25]:
df_effectifs.apply(lambda col: col.duplicated().sum())


annee                  7317447
patho_niv1             7317432
patho_niv2             7317401
patho_niv3             7317388
top                    7317371
cla_age_5              7317429
sexe                   7317447
region                 7317432
dept                   7317349
Ntop                   7307633
Npop                   7310709
prev                   7266931
Niveau prioritaire     7317444
libelle_classe_age     7317429
libelle_sexe           7317447
tri                    7317371
libelle_region         7317432
departement            7317349
libelle_departement    7317349
dtype: int64

### Poids du dataset avant et après suppression des doublons

Avant de nettoyer les données, j’affiche le poids mémoire du DataFrame afin d’évaluer l’impact des doublons sur la taille totale du fichier.  
Après suppression des lignes dupliquées, j’affiche à nouveau le poids du dataset pour mesurer le gain en mémoire et vérifier que le nettoyage a bien été appliqué.


In [26]:
buf = io.StringIO()
df_effectifs.info(buf=buf, verbose=False)
print("Poids AVANT nettoyage :", buf.getvalue().splitlines()[-1])



Poids AVANT nettoyage : memory usage: 1.0 GB


In [27]:
# Suppression des doublons
df_effectifs = df_effectifs.drop_duplicates()


In [28]:
# Afficher uniquement la ligne "memory usage"
buf = io.StringIO()
df_effectifs.info(buf=buf, verbose=False)
print("Poids APRÈS nettoyage :", buf.getvalue().splitlines()[-1])

Poids APRÈS nettoyage : memory usage: 212.1 MB


In [29]:
df_effectifs.head()


,annee,patho_niv1,patho_niv2,patho_niv3,top,cla_age_5,sexe,region,dept,Ntop,Npop,prev,Niveau prioritaire,libelle_classe_age,libelle_sexe,tri,libelle_region,departement,libelle_departement
0,2023,Maladies inflammatoires ou rares ou infection VIH,Maladies rares,Mucoviscidose,RAR_MUC_IND,95et+,9,32,59,NaN,8460,NaN,3,plus de 95 ans,tous sexes,78.0,Hauts-de-France,59,Nord
5,2023,Maladies inflammatoires ou rares ou infection VIH,Maladies rares,Mucoviscidose,RAR_MUC_IND,95et+,9,32,60,NaN,2470,NaN,3,plus de 95 ans,tous sexes,78.0,Hauts-de-France,60,Oise
10,2023,Maladies inflammatoires ou rares ou infection VIH,Maladies rares,Mucoviscidose,RAR_MUC_IND,95et+,9,32,62,NaN,5010,NaN,3,plus de 95 ans,tous sexes,78.0,Hauts-de-France,62,Pas-de-Calais
15,2023,Maladies inflammatoires ou rares ou infection VIH,Maladies rares,Mucoviscidose,RAR_MUC_IND,95et+,9,32,80,NaN,2220,NaN,3,plus de 95 ans,tous sexes,78.0,Hauts-de-France,80,Somme
20,2023,Maladies inflammatoires ou rares ou infection VIH,Maladies rares,Mucoviscidose,RAR_MUC_IND,95et+,9,44,10,NaN,1570,NaN,3,plus de 95 ans,tous sexes,78.0,Grand Est,10,Aube


In [30]:
# Valeurs manquantes
(
    df_effectifs
    .isna()
    .sum(axis=0)
)

annee                       0
patho_niv1                  0
patho_niv2             152712
patho_niv3             330876
top                         0
cla_age_5                   0
sexe                        0
region                      0
dept                        0
Ntop                   390147
Npop                        0
prev                   390147
Niveau prioritaire      19089
libelle_classe_age          0
libelle_sexe                0
tri                     19089
libelle_region              0
departement                 0
libelle_departement         0
dtype: int64

Dans ce jeu de données, les valeurs indiquées comme NaN ne correspondent pas à de véritables valeurs manquantes. Elles apparaissent simplement lorsqu’un département ne présente aucun cas pour une pathologie donnée. Dans ce contexte, il est donc logique de remplacer ces NaN par 0, afin de refléter correctement l’absence de cas observés.

In [31]:
df_effectifs = df_effectifs.fillna(0)


###  Après avoir passer les NAN à 0

In [32]:
# Valeurs manquantes
(
    df_effectifs
    .isna()
    .sum(axis=0)
)

annee                  0
patho_niv1             0
patho_niv2             0
patho_niv3             0
top                    0
cla_age_5              0
sexe                   0
region                 0
dept                   0
Ntop                   0
Npop                   0
prev                   0
Niveau prioritaire     0
libelle_classe_age     0
libelle_sexe           0
tri                    0
libelle_region         0
departement            0
libelle_departement    0
dtype: int64

### Nettoyage et préparation des données

- J’ai décidé de supprimer les parenthèses dans les variables *patho_niv2* et *patho_niv3*, car elles entraînaient un affichage peu lisible dans les visualisations et ajoutaient trop d’informations inutiles.
- Je me suis concentrée sur la France hexagonale, en incluant la Corse et les DOM.
- Les codes **FRANCE** et **Tout département** correspondent à des agrégats régionaux ou nationaux (valeurs non rattachées à un département).  
  Comme ils faussent l’analyse territoriale, j’ai choisi de les supprimer.


In [17]:
# Nettoyage des parenthèses
cols = ["patho_niv2", "patho_niv3"]
for c in cols:
    df_effectifs.loc[:, c] = df_effectifs[c].str.replace(r"\s*\(.*\)", "", regex=True)

# Grand filtre de nettoyage , je veux gardé que la france hexagonale
hors_hexa = ['Tout département', 'Guadeloupe ', 'Haute-Corse', 'Martinique', 'La Réunion', 'Guyane', 'Mayotte', 'Corse-du-Sud','FRANCE']

df_effectifs = df_effectifs[
    (~df_effectifs['libelle_departement'].astype(str).isin(hors_hexa)) &
    
    # On enlève la région "France" qui englobe tout
    (df_effectifs['libelle_region'].astype(str) != 'FRANCE')
]


Pour ne conserver que les observations réellement exploitables, je filtre le dataset
en gardant uniquement les lignes où la variable `prev` est strictement supérieure à 0.
Cela permet d’éliminer les enregistrements sans prévalence et d’alléger les analyses
et visualisations suivantes.

In [33]:
df_effectifs = df_effectifs[df_effectifs["prev"] > 0]


In [34]:
df_effectifs.dtypes


annee                    int64
patho_niv1                 str
patho_niv2              object
patho_niv3              object
top                        str
cla_age_5                  str
sexe                     int64
region                   int64
dept                       str
Ntop                   float64
Npop                     int64
prev                   float64
Niveau prioritaire      object
libelle_classe_age         str
libelle_sexe               str
tri                    float64
libelle_region             str
departement                str
libelle_departement        str
dtype: object

### Nettoyage des colonnes `patho_niv2` et `patho_niv3` après remplacement des `NaN`

Lors du remplacement des valeurs manquantes (`NaN`) par `0`, les colonnes `patho_niv2` et
`patho_niv3` ont changé de type : elles sont passées du type *string* au type *object*.
Lorsqu’une colonne contient un mélange de valeurs c'est ce que pandas fait numériques, textuelles ou des `NaN`, elle est automatiquement convertie en `object`.
Comme les `NaN` avaient été remplacés par `0`, ces valeurs sont restées sous forme de `"0"` après conversion en `str`.  
Je les ai supprimé pour ne conserver que les niveaux de pathologie valides.


In [35]:
df_effectifs["patho_niv2"] = df_effectifs["patho_niv2"].astype(str).str.strip()
df_effectifs["patho_niv3"] = df_effectifs["patho_niv3"].astype(str).str.strip()


In [36]:
df_effectifs.dtypes


annee                    int64
patho_niv1                 str
patho_niv2                 str
patho_niv3                 str
top                        str
cla_age_5                  str
sexe                     int64
region                   int64
dept                       str
Ntop                   float64
Npop                     int64
prev                   float64
Niveau prioritaire      object
libelle_classe_age         str
libelle_sexe               str
tri                    float64
libelle_region             str
departement                str
libelle_departement        str
dtype: object

In [45]:
df_effectifs = df_effectifs[
    (df_effectifs["patho_niv1"] != "0") &
    (df_effectifs["patho_niv2"] != "0") &
    (df_effectifs["patho_niv3"] != "0")
]

### Suppression des colonnes non utilisées

Dans cette étape, je retire les colonnes qui ne sont pas nécessaires pour l’analyse finale.  
Il s’agit principalement de variables techniques, de codes intermédiaires ou de niveaux d’agrégation
qui ne sont pas exploités dans les visualisations (ex. : codes de tri, niveaux prioritaires, variables
de regroupement, ou encore des informations redondantes comme la colonne **region** issue d’un des
datasets utilisés pour la jointure).

L’option `errors='ignore'` permet d’éviter une erreur si certaines colonnes ont déjà été supprimées
lors d’étapes précédentes du nettoyage.


In [46]:
# Suppression des colonnes inutiles pour le nettoyage final
df_effectifs = df_effectifs.drop(
    columns=[
        "Code département",
        "tri",
        "Niveau prioritaire",
        "region",
        "dept",
        "departement",
        "cla_age_5",
        "top",
        "sexe",

    ],
    errors='ignore' # Évite de faire une erreur si ça a déjà été supprimée
)




### Renommage de certaines colonnes

Pour améliorer la lisibilité du dataset et faciliter la compréhension des analyses,
j’ai choisi de renommer plusieurs colonnes.  
Afin d’obtenir des intitulés plus courts, plus explicites et cohérents avec les visualisations produites par la suite.



In [38]:
# Renommage des colonens
df_effectifs = df_effectifs.rename(columns={
    "libelle_departement": "Département",
    "libelle_region": "Région",
    "prev" : "Prévalence",
    "Npop": "Population de référence",
    "Ntop": "Effectif",
    "libelle_sexe" : "Sexe",
    "libelle_classe_age" : "Classe d'age"


})

In [47]:
df_effectifs = df_effectifs.dropna()

In [48]:
df_effectifs

,annee,patho_niv1,patho_niv2,patho_niv3,Effectif,Population de référence,Prévalence,Classe d'age,Sexe,Région,Département
150,2023,Maladies inflammatoires ou rares ou infection VIH,Maladies rares,Mucoviscidose,150.0,981490,0.015,tous âges,hommes,Ile-de-France,Paris
155,2023,Maladies inflammatoires ou rares ou infection VIH,Maladies rares,Mucoviscidose,30.0,204050,0.015,tous âges,hommes,Centre-Val de Loire,Eure-et-Loir
160,2023,Maladies inflammatoires ou rares ou infection VIH,Maladies rares,Mucoviscidose,20.0,123680,0.016,tous âges,hommes,Bourgogne-Franche-Comté,Jura
170,2023,Maladies inflammatoires ou rares ou infection VIH,Maladies rares,Mucoviscidose,40.0,259740,0.014,tous âges,hommes,Bourgogne-Franche-Comté,Saône-et-Loire
175,2023,Maladies inflammatoires ou rares ou infection VIH,Maladies rares,Mucoviscidose,10.0,65260,0.018,tous âges,hommes,Bourgogne-Franche-Comté,Territoire de Belfort
...,...,...,...,...,...,...,...,...,...,...,...
7317415,2023,Maladies cardioneurovasculaires,Artériopathie périphérique,Artériopathie périphérique,40.0,47150,0.091,de 45 à 49 ans,femmes,Pays de la Loire,Loire-Atlantique
7317420,2023,Maladies cardioneurovasculaires,Artériopathie périphérique,Artériopathie périphérique,40.0,21550,0.162,de 45 à 49 ans,femmes,Pays de la Loire,Vendée
7317425,2023,Maladies cardioneurovasculaires,Artériopathie périphérique,Artériopathie périphérique,30.0,17860,0.174,de 45 à 49 ans,femmes,Bretagne,Côtes-d'Armor
7317430,2023,Maladies cardioneurovasculaires,Artériopathie périphérique,Artériopathie périphérique,40.0,34710,0.107,de 45 à 49 ans,femmes,Bretagne,Ille-et-Vilaine


### Exploration des niveaux de pathologies

Avant d’appliquer des filtres plus précis, j’affiche les valeurs uniques de `patho_niv2` et `patho_niv3`
(en excluant les valeurs `NaN`). Cela permet de vérifier la cohérence des libellés et d’identifier
éventuels regroupements ou corrections à effectuer avant l’analyse.


In [49]:
print(df_effectifs['patho_niv1'].dropna().unique())
print(df_effectifs['patho_niv2'].dropna().unique())
print(df_effectifs['patho_niv3'].dropna().unique())


<StringArray>
[                                                               'Maladies inflammatoires ou rares ou infection VIH',
                                                                                           'Maladies neurologiques',
                                                                                          'Maladies psychiatriques',
 'Pas de pathologie repérée, traitement, maternité, hospitalisation ou traitement antalgique ou anti-inflammatoire',
                                                                                   'Total consommants tous régimes',
                                                              'Traitements du risque vasculaire (hors pathologies)',
                                                                      'Traitements psychotropes (hors pathologies)',
                                                                                  'Maladies cardioneurovasculaires',
                                                  

# MISE EN PLACE DU REPORTING EXCEL :

Je vais créer un fichier Excel qui contiendra les données nettoyées dans un onglet "cleaned_data".

Le fichier complet dépasse la limite maximale autorisée par Excel (1 048 576 lignes par feuille).  
Pour éviter toute erreur lors de l’export et garantir un affichage fluide,  
j’ai choisi d’extraire uniquement une partie des données, jusqu’au seuil maximal supporté par Excel.

Cette extraction partielle permet :
- de réduire la taille du fichier final ;
- d’accélérer le chargement dans Excel ;
- de faciliter la création de visualisations ;
- de répartir les données sur plusieurs feuilles si nécessaire, afin de disposer
  d’onglets thématiques pouvant servir de filtres pour les analyses suivantes.

Afin d’obtenir un fichier Excel plus léger, mieux structuré et plus simple à manipuler pour les étapes de reporting et de visualisation.


In [50]:
file_path = '../DATA/Dashboards_Pathologies.xlsx'

# Supprimer le fichier s'il existe déjà
if os.path.exists(file_path):
    os.remove(file_path)

# Réécriture du fichier
with pd.ExcelWriter(file_path) as writer:
    df_effectifs.to_excel(writer, sheet_name='cleaned_data', index=False)

# 2 ème dataset que je vais étudier :

# Dépenses remboursées affectées à chaque pathologie

Les données présentent des informations sur les dépenses remboursées par l’ensemble des régimes d’assurance maladie et affectées à chaque pathologie, traitement chronique ou épisode de soins. Ces dépenses sont réparties en trois grandes catégories :

* les soins de ville (consultations médicales, soins infirmiers, actes de kinésithérapie, médicaments, biologie, transports, etc.)  ; 

* les hospitalisations dans les établissements de santé publics ou privés ;

* les prestations en espèces, notamment les indemnités journalières.

## Deux types d’indicateurs sont proposés pour chacune de ces catégories : 

* les dépenses totales annuelles remboursées, qui mesurent le poids financier global d’une pathologie ;

* les dépenses moyennes annuelles remboursées par patient, qui permettent d’apprécier le coût moyen associé à chaque pathologie.

Ces deux indicateurs offrent ainsi une vision complémentaire des dépenses de santé, en combinant à la fois l’impact global et la charge moyenne par patient.

In [55]:
chunks = []

for chunk in pd.read_csv(
    'https://minio.lab.sspcloud.fr/aissataa/PROJET_MEYTE/depenses.csv',
    sep=';',
    chunksize=100_000,
    low_memory=False
):
    filtered = chunk[
        (chunk['annee'] >= 2021) &
        (chunk['montant'] > 0)
    ]
    chunks.append(filtered)

df_depenses = pd.concat(chunks, ignore_index=True)
df_depenses


,annee,patho_niv1,patho_niv2,patho_niv3,top,dep_niv_1,dep_niv_2,montant,Ntop,N_recourant_au_poste,montant_moy,Niveau prioritaire,tri,type_somme
0,2023,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,ALD_CAT_CAT,Hospitalisations (tous secteurs),Hospitalisations liste en sus MCO secteur priv...,3856836,1714310.0,34840.0,2.0,"1,2,3",16.0,Partiel
1,2023,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,ALD_CAT_CAT,Hospitalisations (tous secteurs),Total hospitalisations (tous secteurs) rembour...,308350458,1714310.0,1232630.0,180.0,"1,2,3",16.0,Total
2,2023,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,ALD_CAT_CAT,Prestations en espèces,Indemnités journalières maladie et AT/MP rembo...,117367790,1714310.0,125760.0,68.0,"1,2,3",16.0,Partiel
3,2023,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,ALD_CAT_CAT,Prestations en espèces,Prestations d'invalidité remboursées,263517389,1714310.0,68580.0,154.0,"1,2,3",16.0,Partiel
4,2023,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,ALD_CAT_CAT,Soins de ville,Autres dépenses de soins de ville remboursés,9004352,1714310.0,424990.0,5.0,"1,2,3",16.0,Partiel
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6401,2022,Traitements psychotropes (hors pathologies),Traitements neuroleptiques (hors pathologies),Traitements neuroleptiques (hors pathologies),TPS_NLE_EXC,Soins de ville,Autres produits de santé remboursés,10878069,335640.0,212200.0,32.0,"2,3",55.0,Partiel
6402,2022,Traitements psychotropes (hors pathologies),Traitements neuroleptiques (hors pathologies),Traitements neuroleptiques (hors pathologies),TPS_NLE_EXC,Soins de ville,Soins de généralistes remboursés,11874138,335640.0,300910.0,35.0,"2,3",55.0,Partiel
6403,2022,Traitements psychotropes (hors pathologies),Traitements neuroleptiques (hors pathologies),Traitements neuroleptiques (hors pathologies),TPS_NLE_EXC,Soins de ville,Soins dentaires remboursés,6030821,335640.0,118450.0,18.0,"2,3",55.0,Partiel
6404,2022,Traitements psychotropes (hors pathologies),Traitements neuroleptiques (hors pathologies),Traitements neuroleptiques (hors pathologies),TPS_NLE_EXC,Soins de ville,Total soins de ville remboursés,199465569,335640.0,335630.0,594.0,"2,3",55.0,Total


# Les champs sont les suivants avant nettoyage:

In [56]:
df_depenses.columns


Index(['annee', 'patho_niv1', 'patho_niv2', 'patho_niv3', 'top', 'dep_niv_1',
       'dep_niv_2', 'montant', 'Ntop', 'N_recourant_au_poste', 'montant_moy',
       'Niveau prioritaire', 'tri', 'type_somme'],
      dtype='str')

In [57]:
# Suppression des colonnes inutiles pour le nettoyage final
df_depenses = df_depenses.drop(
    columns=[
        "tri",
        "Niveau prioritaire",
        "type_somme",
        "top",
        "dep_niv_1"

    ],
    errors='ignore' # Évite de faire une erreur si ça a déjà été supprimée
)




# Nettoyage des données

In [58]:
(
    df_depenses
    .isna()
    .sum(axis=0)
)

annee                      0
patho_niv1                 0
patho_niv2               686
patho_niv3              1496
dep_niv_2                  0
montant                    0
Ntop                      42
N_recourant_au_poste      42
montant_moy               42
dtype: int64

In [59]:
df_depenses = df_depenses.drop_duplicates()


In [60]:
df_depenses


,annee,patho_niv1,patho_niv2,patho_niv3,dep_niv_2,montant,Ntop,N_recourant_au_poste,montant_moy
0,2023,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,Hospitalisations liste en sus MCO secteur priv...,3856836,1714310.0,34840.0,2.0
1,2023,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,Total hospitalisations (tous secteurs) rembour...,308350458,1714310.0,1232630.0,180.0
2,2023,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,Indemnités journalières maladie et AT/MP rembo...,117367790,1714310.0,125760.0,68.0
3,2023,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,Prestations d'invalidité remboursées,263517389,1714310.0,68580.0,154.0
4,2023,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,Autres dépenses de soins de ville remboursés,9004352,1714310.0,424990.0,5.0
...,...,...,...,...,...,...,...,...,...
6401,2022,Traitements psychotropes (hors pathologies),Traitements neuroleptiques (hors pathologies),Traitements neuroleptiques (hors pathologies),Autres produits de santé remboursés,10878069,335640.0,212200.0,32.0
6402,2022,Traitements psychotropes (hors pathologies),Traitements neuroleptiques (hors pathologies),Traitements neuroleptiques (hors pathologies),Soins de généralistes remboursés,11874138,335640.0,300910.0,35.0
6403,2022,Traitements psychotropes (hors pathologies),Traitements neuroleptiques (hors pathologies),Traitements neuroleptiques (hors pathologies),Soins dentaires remboursés,6030821,335640.0,118450.0,18.0
6404,2022,Traitements psychotropes (hors pathologies),Traitements neuroleptiques (hors pathologies),Traitements neuroleptiques (hors pathologies),Total soins de ville remboursés,199465569,335640.0,335630.0,594.0


# Création du fichier Excel

Je vais créer un nouveau fichier Excel qui contiendra les données nettoyées dans un onglet "cleaned_data".

In [92]:
import pandas as pd
from openpyxl import load_workbook
from openpyxl.chart import BarChart, PieChart, LineChart, Reference
from openpyxl.styles import PatternFill, Alignment, Font, Border, Side
from openpyxl.worksheet.datavalidation import DataValidation

# === 1. CHARGEMENT ET EXPORT DE LA SOURCE ===
output = '../DATA/Dashboard.xlsx'

with pd.ExcelWriter(output, engine='openpyxl') as w:
    df_depenses.to_excel(w, sheet_name='cleaned_data', index=False)

wb = load_workbook(output)
ws_data = wb['cleaned_data']

# Détection dynamique des colonnes
cols_mapping = {cell.value: cell.column_letter for cell in ws_data[1]}

c_annee   = cols_mapping.get('annee', 'A')
c_patho   = cols_mapping.get('patho_niv1', 'B')
c_poste   = cols_mapping.get('dep_niv_2', 'E')
c_montant = cols_mapping.get('montant', 'F')
c_recour  = cols_mapping.get('N_recourant_au_poste', 'H')
c_moy     = cols_mapping.get('montant_moy', 'I')

COULEURS = {
    'bleu_fonce': '1F4E78', 'bleu': '2F5496', 'bleu_clair': '4472C4',
    'vert': '70AD47', 'orange': 'C65911', 'jaune': 'FFF2CC'
}
trait = Side(style='thin', color='CCCCCC')
bordure = Border(left=trait, right=trait, top=trait, bottom=trait)

def entete(cell, texte, couleur):
    cell.value = texte
    cell.font = Font(bold=True, size=11, color='FFFFFF')
    cell.fill = PatternFill(start_color=couleur, end_color=couleur, fill_type='solid')
    cell.alignment = Alignment(horizontal='center', vertical='center')
    cell.border = bordure

# === 2. CRÉATION DE L'ONGLET RESSOURCES ===
if 'Ressources' in wb.sheetnames: del wb['Ressources']
ress = wb.create_sheet('Ressources', 0)

annees = sorted(df_depenses['annee'].unique().astype(int))
postes = sorted(df_depenses['dep_niv_2'].dropna().unique())
pathos = sorted(df_depenses['patho_niv1'].unique())

ress['A1'] = 'Annees'
for i, annee in enumerate(annees, 2): ress[f'A{i}'] = int(annee)

ress['B1'] = 'Postes'
for i, poste in enumerate(postes, 2): ress[f'B{i}'] = poste

ress['C1'] = 'Pathologies'
for i, patho in enumerate(pathos, 2): ress[f'C{i}'] = patho

ress.column_dimensions['A'].width = 12
ress.column_dimensions['B'].width = 40
ress.column_dimensions['C'].width = 40

print("✓ Onglet Ressources créé")

# === 3. CRÉATION DE L'ONGLET GRAPHIQUES ===
if 'Graphiques' in wb.sheetnames: del wb['Graphiques']
ws = wb.create_sheet('Graphiques', 1)
ws.sheet_view.showGridLines = True

# TITRE
ws['A1'] = "CARTOGRAPHIE DES DÉPENSES PAR PATHOLOGIE"
ws['A1'].font = Font(bold=True, size=16, color='FFFFFF')
ws['A1'].fill = PatternFill(start_color=COULEURS['bleu_fonce'], end_color=COULEURS['bleu_fonce'], fill_type='solid')
ws.merge_cells('A1:N1')
ws.row_dimensions[1].height = 35
ws['A1'].alignment = Alignment(horizontal='center', vertical='center')

# FILTRES
ws['A3'] = 'Année :'
ws['A3'].font = Font(bold=True, size=11)
ws['B3'].fill = PatternFill(start_color=COULEURS['jaune'], end_color=COULEURS['jaune'], fill_type='solid')
ws['B3'].border = bordure
dv_annee = DataValidation(type='list', formula1=f"=Ressources!$A$2:$A${len(annees) + 1}")
ws.add_data_validation(dv_annee)
dv_annee.add('B3')
ws['B3'] = int(annees[-1])

ws['C3'] = 'Poste Dépense :'
ws['C3'].font = Font(bold=True, size=11)
ws['D3'].fill = PatternFill(start_color=COULEURS['jaune'], end_color=COULEURS['jaune'], fill_type='solid')
ws['D3'].border = bordure
dv_poste = DataValidation(type='list', formula1=f"=Ressources!$B$2:$B${len(postes) + 1}")
ws.add_data_validation(dv_poste)
dv_poste.add('D3')
ws['D3'] = postes[0]

# === 4. CRÉATION DE L'ONGLET DE CALCULS ===
if 'Calculs' in wb.sheetnames: del wb['Calculs']
ws_calc = wb.create_sheet('Calculs', 2)

# ✅ G1 : TOP PATHOLOGIES (Filtre Année ET Poste)
entete(ws_calc['A1'], 'Pathologie', COULEURS['bleu_clair'])
entete(ws_calc['B1'], 'Montant', COULEURS['bleu_clair'])
for i, patho in enumerate(pathos, 2):
    ws_calc.cell(row=i, column=1, value=patho)
    ws_calc.cell(row=i, column=2, value=f"=SUMIFS(cleaned_data!${c_montant}:${c_montant}, cleaned_data!${c_annee}:${c_annee}, Graphiques!$B$3, cleaned_data!${c_poste}:${c_poste}, Graphiques!$D$3, cleaned_data!${c_patho}:${c_patho}, A{i})")
    ws_calc.cell(row=i, column=2).number_format = '#,##0'

# ✅ G2 : RÉPARTITION PATHOLOGIES (Filtre Année ET Poste) - POUR LE PIE CHART
entete(ws_calc['D1'], 'Pathologie', COULEURS['bleu'])
entete(ws_calc['E1'], 'Montant', COULEURS['bleu'])
for i, patho in enumerate(pathos, 2):
    ws_calc.cell(row=i, column=4, value=patho)
    ws_calc.cell(row=i, column=5, value=f"=SUMIFS(cleaned_data!${c_montant}:${c_montant}, cleaned_data!${c_annee}:${c_annee}, Graphiques!$B$3, cleaned_data!${c_poste}:${c_poste}, Graphiques!$D$3, cleaned_data!${c_patho}:${c_patho}, D{i})")
    ws_calc.cell(row=i, column=5).number_format = '#,##0'

# ✅ G3 : RÉPARTITION PAR POSTE (Filtre Année SEULEMENT)
entete(ws_calc['G1'], 'Poste', COULEURS['orange'])
entete(ws_calc['H1'], 'Montant', COULEURS['orange'])
for i, poste in enumerate(postes, 2):
    ws_calc.cell(row=i, column=7, value=poste)
    ws_calc.cell(row=i, column=8, value=f"=SUMIFS(cleaned_data!${c_montant}:${c_montant}, cleaned_data!${c_annee}:${c_annee}, Graphiques!$B$3, cleaned_data!${c_poste}:${c_poste}, G{i})")
    ws_calc.cell(row=i, column=8).number_format = '#,##0'

# G4 : Coût Moyen (Filtre Année ET Poste)
entete(ws_calc['J1'], 'Pathologie', COULEURS['bleu'])
entete(ws_calc['K1'], 'Montant Moyen', COULEURS['bleu'])
for i, patho in enumerate(pathos, 2):
    ws_calc.cell(row=i, column=10, value=patho)
    ws_calc.cell(row=i, column=11, value=f"=AVERAGEIFS(cleaned_data!${c_moy}:${c_moy}, cleaned_data!${c_annee}:${c_annee}, Graphiques!$B$3, cleaned_data!${c_poste}:${c_poste}, Graphiques!$D$3, cleaned_data!${c_patho}:${c_patho}, J{i})")
    ws_calc.cell(row=i, column=11).number_format = '#,##0.00'

# G5 : Évolution Historique (Filtre Poste SEULEMENT)
entete(ws_calc['M1'], 'Année', COULEURS['vert'])
entete(ws_calc['N1'], 'Montant', COULEURS['vert'])
for i, annee in enumerate(annees, 2):
    ws_calc.cell(row=i, column=13, value=int(annee))
    ws_calc.cell(row=i, column=14, value=f"=SUMIFS(cleaned_data!${c_montant}:${c_montant}, cleaned_data!${c_annee}:${c_annee}, M{i}, cleaned_data!${c_poste}:${c_poste}, Graphiques!$D$3)")
    ws_calc.cell(row=i, column=14).number_format = '#,##0'

# G6 : Recourants (Filtre Année ET Poste)
entete(ws_calc['P1'], 'Pathologie', COULEURS['orange'])
entete(ws_calc['Q1'], 'Recourants', COULEURS['orange'])
for i, patho in enumerate(pathos, 2):
    ws_calc.cell(row=i, column=16, value=patho)
    ws_calc.cell(row=i, column=17, value=f"=SUMIFS(cleaned_data!${c_recour}:${c_recour}, cleaned_data!${c_annee}:${c_annee}, Graphiques!$B$3, cleaned_data!${c_poste}:${c_poste}, Graphiques!$D$3, cleaned_data!${c_patho}:${c_patho}, P{i})")
    ws_calc.cell(row=i, column=17).number_format = '#,##0'

print("✓ Formules dynamiques calculées")

# === 5. AJOUT DES GRAPHIQUES ===

# G1: BARRES DES PATHOLOGIES
chart1 = BarChart()
chart1.type = "col"
chart1.title = "Dépenses par Pathologie (Année & Poste)"
chart1.y_axis.title = 'Montant (€)'
chart1.height = 13
chart1.width = 17
chart1.legend = None
chart1.add_data(Reference(ws_calc, min_col=2, min_row=1, max_row=1 + len(pathos)), titles_from_data=True)
chart1.set_categories(Reference(ws_calc, min_col=1, min_row=2, max_row=1 + len(pathos)))
ws.add_chart(chart1, "A5")

# ✅ G2: PIE CHART DES PATHOLOGIES (Colonnes D-E, filtre année + poste)
chart2 = PieChart()
chart2.title = "Répartition des Pathologies (Année & Poste)"
chart2.height = 13
chart2.width = 15
chart2.add_data(Reference(ws_calc, min_col=5, min_row=1, max_row=1 + len(pathos)), titles_from_data=True)
chart2.set_categories(Reference(ws_calc, min_col=4, min_row=2, max_row=1 + len(pathos)))
ws.add_chart(chart2, "K5")

# G3: HISTOGRAMME POSTES
chart3 = BarChart()
chart3.type = "col"
chart3.title = "Dépenses Totales par Poste de Santé"
chart3.height = 13
chart3.width = 17
chart3.legend = None
chart3.add_data(Reference(ws_calc, min_col=8, min_row=1, max_row=1 + len(postes)), titles_from_data=True)
chart3.set_categories(Reference(ws_calc, min_col=7, min_row=2, max_row=1 + len(postes)))
ws.add_chart(chart3, "A25")

# G4: COÛT MOYEN PAR PATIENT
chart4 = BarChart()
chart4.type = "col"
chart4.title = "Coût Moyen Annuel par Patient (€)"
chart4.height = 13
chart4.width = 15
chart4.legend = None
chart4.add_data(Reference(ws_calc, min_col=11, min_row=1, max_row=1 + len(pathos)), titles_from_data=True)
chart4.set_categories(Reference(ws_calc, min_col=10, min_row=2, max_row=1 + len(pathos)))
ws.add_chart(chart4, "K25")

# G5: COURBE ÉVOLUTION CHRONOLOGIQUE
chart5 = LineChart()
chart5.title = "Évolution Pluri-annuelle des Montants"
chart5.height = 13
chart5.width = 17
chart5.add_data(Reference(ws_calc, min_col=14, min_row=1, max_row=1 + len(annees)), titles_from_data=True)
chart5.set_categories(Reference(ws_calc, min_col=13, min_row=2, max_row=1 + len(annees)))
ws.add_chart(chart5, "A45")

# G6: EFFECTIFS DES PATIENTS RECOURANTS
chart6 = BarChart()
chart6.type = "col"
chart6.title = "Volume de Patients Pris en Charge"
chart6.height = 13
chart6.width = 15
chart6.legend = None
chart6.add_data(Reference(ws_calc, min_col=17, min_row=1, max_row=1 + len(pathos)), titles_from_data=True)
chart6.set_categories(Reference(ws_calc, min_col=16, min_row=2, max_row=1 + len(pathos)))
ws.add_chart(chart6, "K45")

print("✓ 6 Graphiques créés")

# === 6. NETTOYAGE ===
ws_calc.sheet_state = 'hidden'

ws.column_dimensions['A'].width = 15
ws.column_dimensions['B'].width = 22
ws.column_dimensions['C'].width = 22
ws.column_dimensions['D'].width = 45

ws.sheet_view.showGridLines = False

# Sauvegarde
wb.save(output)
wb.close()

print("\n🚀 DASHBOARD CORRIGÉ : PIE CHART PAR PATHOLOGIE !")

✓ Onglet Ressources créé
✓ Formules dynamiques calculées
✓ 6 Graphiques créés

🚀 DASHBOARD CORRIGÉ : PIE CHART PAR PATHOLOGIE !


In [89]:
import pandas as pd
from openpyxl import load_workbook
from openpyxl.chart import BarChart, PieChart, LineChart, Reference
from openpyxl.styles import PatternFill, Alignment, Font, Border, Side
from openpyxl.worksheet.datavalidation import DataValidation

# === 1. CHARGEMENT ET EXPORT DE LA SOURCE ===
output = '../DATA/Dashboard.xlsx'

with pd.ExcelWriter(output, engine='openpyxl') as w:
    df_depenses.to_excel(w, sheet_name='cleaned_data', index=False)

wb = load_workbook(output)
ws_data = wb['cleaned_data']

# Détection dynamique des colonnes
cols_mapping = {cell.value: cell.column_letter for cell in ws_data[1]}

c_annee   = cols_mapping.get('annee', 'A')
c_patho   = cols_mapping.get('patho_niv1', 'B')
c_poste   = cols_mapping.get('dep_niv_2', 'E')
c_montant = cols_mapping.get('montant', 'F')
c_recour  = cols_mapping.get('N_recourant_au_poste', 'H')
c_moy     = cols_mapping.get('montant_moy', 'I')

COULEURS = {
    'bleu_fonce': '1F4E78', 'bleu': '2F5496', 'bleu_clair': '4472C4',
    'vert': '70AD47', 'orange': 'C65911', 'jaune': 'FFF2CC'
}
trait = Side(style='thin', color='CCCCCC')
bordure = Border(left=trait, right=trait, top=trait, bottom=trait)

def entete(cell, texte, couleur):
    cell.value = texte
    cell.font = Font(bold=True, size=11, color='FFFFFF')
    cell.fill = PatternFill(start_color=couleur, end_color=couleur, fill_type='solid')
    cell.alignment = Alignment(horizontal='center', vertical='center')
    cell.border = bordure

# === 2. CRÉATION DE L'ONGLET RESSOURCES ===
if 'Ressources' in wb.sheetnames: del wb['Ressources']
ress = wb.create_sheet('Ressources', 0)

annees = sorted(df_depenses['annee'].unique().astype(int))
postes = sorted(df_depenses['dep_niv_2'].dropna().unique())
pathos = sorted(df_depenses['patho_niv1'].unique())

ress['A1'] = 'Annees'
for i, annee in enumerate(annees, 2): ress[f'A{i}'] = int(annee)

ress['B1'] = 'Postes'
for i, poste in enumerate(postes, 2): ress[f'B{i}'] = poste

ress['C1'] = 'Pathologies'
for i, patho in enumerate(pathos, 2): ress[f'C{i}'] = patho

ress.column_dimensions['A'].width = 12
ress.column_dimensions['B'].width = 40
ress.column_dimensions['C'].width = 40

print("✓ Onglet Ressources créé")

# === 3. CRÉATION DE L'ONGLET GRAPHIQUES ===
if 'Graphiques' in wb.sheetnames: del wb['Graphiques']
ws = wb.create_sheet('Graphiques', 1)
ws.sheet_view.showGridLines = True

# TITRE
ws['A1'] = "CARTOGRAPHIE DES DÉPENSES PAR PATHOLOGIE"
ws['A1'].font = Font(bold=True, size=16, color='FFFFFF')
ws['A1'].fill = PatternFill(start_color=COULEURS['bleu_fonce'], end_color=COULEURS['bleu_fonce'], fill_type='solid')
ws.merge_cells('A1:N1')
ws.row_dimensions[1].height = 35
ws['A1'].alignment = Alignment(horizontal='center', vertical='center')

# FILTRES
ws['A3'] = 'Année :'
ws['A3'].font = Font(bold=True, size=11)
ws['B3'].fill = PatternFill(start_color=COULEURS['jaune'], end_color=COULEURS['jaune'], fill_type='solid')
ws['B3'].border = bordure
dv_annee = DataValidation(type='list', formula1=f"=Ressources!$A$2:$A${len(annees) + 1}")
ws.add_data_validation(dv_annee)
dv_annee.add('B3')
ws['B3'] = int(annees[-1])

ws['C3'] = 'Poste Dépense :'
ws['C3'].font = Font(bold=True, size=11)
ws['D3'].fill = PatternFill(start_color=COULEURS['jaune'], end_color=COULEURS['jaune'], fill_type='solid')
ws['D3'].border = bordure
dv_poste = DataValidation(type='list', formula1=f"=Ressources!$B$2:$B${len(postes) + 1}")
ws.add_data_validation(dv_poste)
dv_poste.add('D3')
ws['D3'] = postes[0]

# === 4. CRÉATION DE L'ONGLET DE CALCULS ===
if 'Calculs' in wb.sheetnames: del wb['Calculs']
ws_calc = wb.create_sheet('Calculs', 2)

# ✅ G1 : TOP PATHOLOGIES (Filtre Année ET Poste)
entete(ws_calc['A1'], 'Pathologie', COULEURS['bleu_clair'])
entete(ws_calc['B1'], 'Montant', COULEURS['bleu_clair'])
for i, patho in enumerate(pathos, 2):
    ws_calc.cell(row=i, column=1, value=patho)
    ws_calc.cell(row=i, column=2, value=f"=SUMIFS(cleaned_data!${c_montant}:${c_montant}, cleaned_data!${c_annee}:${c_annee}, Graphiques!$B$3, cleaned_data!${c_poste}:${c_poste}, Graphiques!$D$3, cleaned_data!${c_patho}:${c_patho}, A{i})")
    ws_calc.cell(row=i, column=2).number_format = '#,##0'

# ✅ G2 : RÉPARTITION PATHOLOGIES (Filtre Année ET Poste) - POUR LE PIE CHART
entete(ws_calc['D1'], 'Pathologie', COULEURS['bleu'])
entete(ws_calc['E1'], 'Montant', COULEURS['bleu'])
for i, patho in enumerate(pathos, 2):
    ws_calc.cell(row=i, column=4, value=patho)
    ws_calc.cell(row=i, column=5, value=f"=SUMIFS(cleaned_data!${c_montant}:${c_montant}, cleaned_data!${c_annee}:${c_annee}, Graphiques!$B$3, cleaned_data!${c_poste}:${c_poste}, Graphiques!$D$3, cleaned_data!${c_patho}:${c_patho}, D{i})")
    ws_calc.cell(row=i, column=5).number_format = '#,##0'

# ✅ G3 : RÉPARTITION PAR POSTE (Filtre Année SEULEMENT)
entete(ws_calc['G1'], 'Poste', COULEURS['orange'])
entete(ws_calc['H1'], 'Montant', COULEURS['orange'])
for i, poste in enumerate(postes, 2):
    ws_calc.cell(row=i, column=7, value=poste)
    ws_calc.cell(row=i, column=8, value=f"=SUMIFS(cleaned_data!${c_montant}:${c_montant}, cleaned_data!${c_annee}:${c_annee}, Graphiques!$B$3, cleaned_data!${c_poste}:${c_poste}, G{i})")
    ws_calc.cell(row=i, column=8).number_format = '#,##0'

# G4 : Coût Moyen (Filtre Année ET Poste)
entete(ws_calc['J1'], 'Pathologie', COULEURS['bleu'])
entete(ws_calc['K1'], 'Montant Moyen', COULEURS['bleu'])
for i, patho in enumerate(pathos, 2):
    ws_calc.cell(row=i, column=10, value=patho)
    ws_calc.cell(row=i, column=11, value=f"=AVERAGEIFS(cleaned_data!${c_moy}:${c_moy}, cleaned_data!${c_annee}:${c_annee}, Graphiques!$B$3, cleaned_data!${c_poste}:${c_poste}, Graphiques!$D$3, cleaned_data!${c_patho}:${c_patho}, J{i})")
    ws_calc.cell(row=i, column=11).number_format = '#,##0.00'

# G5 : Évolution Historique (Filtre Poste SEULEMENT)
entete(ws_calc['M1'], 'Année', COULEURS['vert'])
entete(ws_calc['N1'], 'Montant', COULEURS['vert'])
for i, annee in enumerate(annees, 2):
    ws_calc.cell(row=i, column=13, value=int(annee))
    ws_calc.cell(row=i, column=14, value=f"=SUMIFS(cleaned_data!${c_montant}:${c_montant}, cleaned_data!${c_annee}:${c_annee}, M{i}, cleaned_data!${c_poste}:${c_poste}, Graphiques!$D$3)")
    ws_calc.cell(row=i, column=14).number_format = '#,##0'

# G6 : Recourants (Filtre Année ET Poste)
entete(ws_calc['P1'], 'Pathologie', COULEURS['orange'])
entete(ws_calc['Q1'], 'Recourants', COULEURS['orange'])
for i, patho in enumerate(pathos, 2):
    ws_calc.cell(row=i, column=16, value=patho)
    ws_calc.cell(row=i, column=17, value=f"=SUMIFS(cleaned_data!${c_recour}:${c_recour}, cleaned_data!${c_annee}:${c_annee}, Graphiques!$B$3, cleaned_data!${c_poste}:${c_poste}, Graphiques!$D$3, cleaned_data!${c_patho}:${c_patho}, P{i})")
    ws_calc.cell(row=i, column=17).number_format = '#,##0'

print("✓ Formules dynamiques calculées")

# === 5. AJOUT DES GRAPHIQUES ===

# G1: BARRES DES PATHOLOGIES
chart1 = BarChart()
chart1.type = "col"
chart1.title = "Dépenses par Pathologie (Année & Poste)"
chart1.y_axis.title = 'Montant (€)'
chart1.height = 13
chart1.width = 17
chart1.legend = None
chart1.add_data(Reference(ws_calc, min_col=2, min_row=1, max_row=1 + len(pathos)), titles_from_data=True)
chart1.set_categories(Reference(ws_calc, min_col=1, min_row=2, max_row=1 + len(pathos)))
ws.add_chart(chart1, "A5")

# ✅ G2: PIE CHART DES PATHOLOGIES (Colonnes D-E, filtre année + poste)
chart2 = PieChart()
chart2.title = "Répartition des Pathologies (Année & Poste)"
chart2.height = 13
chart2.width = 15
chart2.add_data(Reference(ws_calc, min_col=5, min_row=1, max_row=1 + len(pathos)), titles_from_data=True)
chart2.set_categories(Reference(ws_calc, min_col=4, min_row=2, max_row=1 + len(pathos)))
ws.add_chart(chart2, "K5")

# G3: HISTOGRAMME POSTES
chart3 = BarChart()
chart3.type = "col"
chart3.title = "Dépenses Totales par Poste de Santé"
chart3.height = 13
chart3.width = 17
chart3.legend = None
chart3.add_data(Reference(ws_calc, min_col=8, min_row=1, max_row=1 + len(postes)), titles_from_data=True)
chart3.set_categories(Reference(ws_calc, min_col=7, min_row=2, max_row=1 + len(postes)))
ws.add_chart(chart3, "A25")

# G4: COÛT MOYEN PAR PATIENT
chart4 = BarChart()
chart4.type = "col"
chart4.title = "Coût Moyen Annuel par Patient (€)"
chart4.height = 13
chart4.width = 15
chart4.legend = None
chart4.add_data(Reference(ws_calc, min_col=11, min_row=1, max_row=1 + len(pathos)), titles_from_data=True)
chart4.set_categories(Reference(ws_calc, min_col=10, min_row=2, max_row=1 + len(pathos)))
ws.add_chart(chart4, "K25")

# G5: COURBE ÉVOLUTION CHRONOLOGIQUE
chart5 = LineChart()
chart5.title = "Évolution Pluri-annuelle des Montants"
chart5.height = 13
chart5.width = 17
chart5.add_data(Reference(ws_calc, min_col=14, min_row=1, max_row=1 + len(annees)), titles_from_data=True)
chart5.set_categories(Reference(ws_calc, min_col=13, min_row=2, max_row=1 + len(annees)))
ws.add_chart(chart5, "A45")

# G6: EFFECTIFS DES PATIENTS RECOURANTS
chart6 = BarChart()
chart6.type = "col"
chart6.title = "Volume de Patients Pris en Charge"
chart6.height = 13
chart6.width = 15
chart6.legend = None
chart6.add_data(Reference(ws_calc, min_col=17, min_row=1, max_row=1 + len(pathos)), titles_from_data=True)
chart6.set_categories(Reference(ws_calc, min_col=16, min_row=2, max_row=1 + len(pathos)))
ws.add_chart(chart6, "K45")

print("✓ 6 Graphiques créés")

# === 6. NETTOYAGE ===
ws_calc.sheet_state = 'hidden'

ws.column_dimensions['A'].width = 15
ws.column_dimensions['B'].width = 22
ws.column_dimensions['C'].width = 22
ws.column_dimensions['D'].width = 45

ws.sheet_view.showGridLines = False

# Sauvegarde
wb.save(output)
wb.close()

print("\n🚀 DASHBOARD CORRIGÉ : PIE CHART PAR PATHOLOGIE !")

✓ Onglet Ressources créé
✓ Formules dynamiques calculées
✓ 6 Graphiques créés

🚀 DASHBOARD CORRIGÉ : PIE CHART PAR PATHOLOGIE !


✓ Onglet Ressources créé avec succès
✓ Formules dynamiques calculées sur l'onglet dédié

🚀 DASHBOARD COMPLÈTEMENT CORRIGÉ ET OPÉRATIONNEL !
